# Cell 1 — Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
from optuna.visualization import (
    plot_optimization_history, plot_param_importances,
)

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 25
DATA_DIR    = Path.cwd().parent / 'data' / 'processed'
RESULTS_DIR = Path.cwd().parent / 'outputs' / 'results'
STUDY_DIR   = Path.cwd().parent / 'outputs' / 'optuna_studies'
FIG_DIR     = Path.cwd().parent / 'outputs' / 'figures'
MODELS_DIR  = Path.cwd().parent / 'models'
for d in [RESULTS_DIR, STUDY_DIR, FIG_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Silence Optuna's per-trial log spam
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Cell 2 — Load splits, define preprocessor, sampling helpers

In [2]:
X_train = pd.read_parquet(DATA_DIR / 'X_train.parquet')
y_train = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']

y_train_int, class_labels = pd.factorize(y_train, sort=True)
y_train_int = pd.Series(y_train_int, index=y_train.index)

numeric_cols     = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"X_train: {X_train.shape}")
print(f"Class labels: {list(class_labels)}")

X_train: (370466, 21)
Class labels: ['With dead victims', 'With injured victims', 'Without victims']


# Cell 3 — Objective for XGBoost + RandomOverSampler

In [3]:
def objective_xgb_ros(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 1.0),
    }
    pipe = ImbPipeline([
        ('pre',     preprocessor),
        ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
        ('clf',     XGBClassifier(tree_method='hist', n_jobs=-1,
                                  random_state=RANDOM_STATE, verbosity=0, **params)),
    ])
    scores = cross_val_score(pipe, X_train, y_train_int,
                             cv=cv, scoring='f1_macro', n_jobs=1)
    return scores.mean()

# Cell 4 — Objective for XGBoost + ClassWeight

In [4]:
def objective_xgb_cw(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 1.0),
    }
    fold_scores = []
    for tr_idx, va_idx in cv.split(X_train, y_train_int):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train_int.iloc[tr_idx], y_train_int.iloc[va_idx]

        pipe = ImbPipeline([
            ('pre', preprocessor),
            ('clf', XGBClassifier(tree_method='hist', n_jobs=-1,
                                  random_state=RANDOM_STATE, verbosity=0, **params)),
        ])
        sw = compute_sample_weight('balanced', y_tr)
        pipe.fit(X_tr, y_tr, clf__sample_weight=sw)
        fold_scores.append(f1_score(y_va, pipe.predict(X_va),
                                    average='macro', zero_division=0))
    return np.mean(fold_scores)

# Cell 5 — Objective for CatBoost + RandomOverSampler

In [ ]:
def objective_cat_ros(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 100, 800),
        'depth':               trial.suggest_int('depth', 4, 10),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
    pipe = ImbPipeline([
        ('pre',     preprocessor),
        ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),  
        ('clf',     CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, **params)),
    ])
    scores = cross_val_score(pipe, X_train, y_train,
                             cv=cv, scoring='f1_macro', n_jobs=1)
    return scores.mean()

# Cell 6 — Run 3 Optuna studies with per-trial progress logging


In [6]:
def run_study(name, objective, n_trials):
    print(f"\nStudy: {name}  ({n_trials} trials)")
    t0 = time.time()

    study = optuna.create_study(
        study_name    = name,
        storage       = f'sqlite:///{STUDY_DIR}/{name}.db',
        direction     = 'maximize',
        sampler       = optuna.samplers.TPESampler(seed=RANDOM_STATE),
        load_if_exists= True,
    )

    already_done = len(study.trials)
    remaining    = n_trials - already_done
    if already_done > 0:
        print(f"  Resuming: {already_done} trials already done, {remaining} to go")

    def log_progress(study, trial):
        print(f"  [{trial.number+1:3d}/{n_trials}] "
              f"macro-F1 = {trial.value:.4f}  "
              f"(best = {study.best_value:.4f})")

    study.optimize(objective, n_trials=remaining,
                   callbacks=[log_progress] if remaining > 0 else [])

    elapsed = (time.time() - t0) / 60
    print(f"\n  Study done. Best macro-F1 = {study.best_value:.4f}  |  {elapsed:.1f} min")
    return study

study_xgb_ros = run_study('xgb_randomover',  objective_xgb_ros, n_trials=50)
study_xgb_cw  = run_study('xgb_classweight', objective_xgb_cw,  n_trials=50)
study_cat_ros = run_study('cat_randomover',  objective_cat_ros, n_trials=50)


Study: xgb_randomover  (50 trials)
  [  1/50] macro-F1 = 0.4830  (best = 0.4830)
  [  2/50] macro-F1 = 0.4768  (best = 0.4830)
  [  3/50] macro-F1 = 0.4814  (best = 0.4830)
  [  4/50] macro-F1 = 0.4747  (best = 0.4830)
  [  5/50] macro-F1 = 0.4780  (best = 0.4830)
  [  6/50] macro-F1 = 0.4739  (best = 0.4830)
  [  7/50] macro-F1 = 0.4727  (best = 0.4830)
  [  8/50] macro-F1 = 0.4950  (best = 0.4950)
  [  9/50] macro-F1 = 0.4810  (best = 0.4950)
  [ 10/50] macro-F1 = 0.4820  (best = 0.4950)
  [ 11/50] macro-F1 = 0.5135  (best = 0.5135)
  [ 12/50] macro-F1 = 0.5165  (best = 0.5165)
  [ 13/50] macro-F1 = 0.5134  (best = 0.5165)
  [ 14/50] macro-F1 = 0.5134  (best = 0.5165)
  [ 15/50] macro-F1 = 0.5058  (best = 0.5165)
  [ 16/50] macro-F1 = 0.4989  (best = 0.5165)
  [ 17/50] macro-F1 = 0.5055  (best = 0.5165)
  [ 18/50] macro-F1 = 0.4993  (best = 0.5165)
  [ 19/50] macro-F1 = 0.4947  (best = 0.5165)
  [ 20/50] macro-F1 = 0.5047  (best = 0.5165)
  [ 21/50] macro-F1 = 0.4752  (best = 0.5165

# Cell 7 — Consolidate all trials from all 3 studies into one CSV

In [7]:
all_trials = []
for study, combo_name in [
    (study_xgb_ros, 'XGBoost_RandomOver'),
    (study_xgb_cw,  'XGBoost_ClassWeight'),
    (study_cat_ros, 'CatBoost_RandomOver'),
]:
    df = study.trials_dataframe(attrs=('number', 'value', 'params', 'duration'))
    df.insert(0, 'combo', combo_name)
    all_trials.append(df)

trials_df = pd.concat(all_trials, ignore_index=True)
trials_df.to_csv(RESULTS_DIR / 'phase4_optuna_trials.csv', index=False)

print(f"Saved {len(trials_df)} trials to phase4_optuna_trials.csv")
print(f"\nBest per combo:")
best_per_combo = (trials_df
                  .sort_values('value', ascending=False)
                  .groupby('combo').first()
                  .reset_index()
                  .sort_values('value', ascending=False)
                  [['combo', 'number', 'value', 'duration']])
print(best_per_combo.to_string(index=False))

best_per_combo.to_csv(RESULTS_DIR / 'phase4_best_configs.csv', index=False)

Saved 150 trials to phase4_optuna_trials.csv

Best per combo:
              combo  number    value               duration
 XGBoost_RandomOver      43 0.518852 0 days 00:03:21.076032
XGBoost_ClassWeight      49 0.518128 0 days 00:02:06.738009
CatBoost_RandomOver      48 0.501578 0 days 00:13:28.787752


# Cell 8 — Refit best hyperparameters on full training set

In [ ]:
def refit_best(study, combo_name):
    p = dict(study.best_params)

    use_sampler = 'RandomOver' in combo_name

    if combo_name.startswith('XGBoost'):
        clf = XGBClassifier(tree_method='hist', n_jobs=-1,
                            random_state=RANDOM_STATE, verbosity=0, **p)
        y_for_fit = y_train_int
    else:
        clf = CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, **p)
        y_for_fit = y_train

    steps = [('pre', preprocessor)]
    if use_sampler:
        steps.append(('sampler', RandomOverSampler(random_state=RANDOM_STATE)))
    steps.append(('clf', clf))
    pipe = ImbPipeline(steps)

    if combo_name == 'XGBoost_ClassWeight':
        sw = compute_sample_weight('balanced', y_for_fit)
        pipe.fit(X_train, y_for_fit, clf__sample_weight=sw)
    else:
        pipe.fit(X_train, y_for_fit)

    joblib.dump(pipe, MODELS_DIR / f'phase4_best_{combo_name}.joblib')
    print(f"{combo_name}: saved  |  best CV macro-F1 = {study.best_value:.4f}")

refit_best(study_xgb_ros, 'XGBoost_RandomOver')
refit_best(study_xgb_cw,  'XGBoost_ClassWeight')
refit_best(study_cat_ros, 'CatBoost_RandomOver')

XGBoost_RandomOver: saved  |  best CV macro-F1 = 0.5189
XGBoost_ClassWeight: saved  |  best CV macro-F1 = 0.5181
CatBoost_RandomOver: saved  |  best CV macro-F1 = 0.5016


# Cell 9 — Save 2 diagnostic plots per study

In [12]:
for study, name in [
    (study_xgb_ros, 'XGBoost_RandomOver'),
    (study_xgb_cw,  'XGBoost_ClassWeight'),
    (study_cat_ros, 'CatBoost_RandomOver'),
]:
    fig_history = plot_optimization_history(study)
    fig_history.update_layout(title=f'{name} — optimisation history')
    fig_history.write_html(FIG_DIR / f'phase4_{name}_history.html')

    fig_importance = plot_param_importances(study)
    fig_importance.update_layout(title=f'{name} — parameter importance')
    fig_importance.write_html(FIG_DIR / f'phase4_{name}_importance.html')

    print(f"  saved: outputs/figures/phase4_{name}_history.html")
    print(f"  saved: outputs/figures/phase4_{name}_importance.html")

  saved: outputs/figures/phase4_XGBoost_RandomOver_history.html
  saved: outputs/figures/phase4_XGBoost_RandomOver_importance.html
  saved: outputs/figures/phase4_XGBoost_ClassWeight_history.html
  saved: outputs/figures/phase4_XGBoost_ClassWeight_importance.html
  saved: outputs/figures/phase4_CatBoost_RandomOver_history.html
  saved: outputs/figures/phase4_CatBoost_RandomOver_importance.html
